# Week 5 – CYP2C8 / CYP2C9 / CYP2C19: Illumina vs PacBio on hg38 chr10

This notebook does the following:

1. Downloads the **hg38/GRCh38 chr10** reference sequence (single FASTA) and indexes it.
2. Downloads the provided **Illumina (interleaved) and PacBio** FASTQ files.
3. Aligns both to chr10 using **minimap2** and creates sorted, indexed BAM files.
4. Calls variants with **bcftools** for each technology.
5. Subsets variants to the CYP2C region (CYP2C19, CYP2C9, CYP2C8) on chr10.
6. Phases the CYP2C variants with **WhatsHap**.
7. Compares phased VCFs between Illumina and PacBio (shared vs technology-specific variants).
8. Counts variants in each gene and prints example phased genotypes.

> Assumes the following tools are already installed and on `$PATH` (via your conda env / CI):
> `minimap2`, `samtools`, `bcftools`, `seqkit`, `whatshap`.


In [1]:
!curl -L http://hgdownload.soe.ucsc.edu/goldenPath/hg38/chromosomes/chr10.fa.gz -o chr10.fa.gz


  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100 41.1M  100 41.1M    0     0  12.3M      0  0:00:03  0:00:03 --:--:-- 12.3M


In [2]:
!gunzip -c chr10.fa.gz > chr10.fa

# samtools / minimap2 indexes
!samtools faidx chr10.fa
!minimap2 -d chr10.mmi chr10.fa


[M::mm_idx_gen::2.198*0.94] collected minimizers
[M::mm_idx_gen::2.745*1.34] sorted minimizers
[M::main::3.803*1.17] loaded/built the index for 1 target sequence(s)
[M::mm_idx_stat] kmer size: 15; skip: 10; is_hpc: 0; #seq: 1
[M::mm_idx_stat::3.956*1.16] distinct minimizers: 16061920 (79.91% are singletons); average occurrences: 1.563; average spacing: 5.329; total length: 133797422
[M::main] Version: 2.30-r1287
[M::main] CMD: minimap2 -d chr10.mmi chr10.fa
[M::main] Real time: 4.009 sec; CPU: 4.645 sec; Peak RSS: 1.266 GB


In [8]:
# Illumina (interleaved paired-end)
!curl -L https://github.com/inumanag/fall25-csc-bioinf/raw/refs/heads/main/week4/data/illumina.fq.bz2 -o illumina.fq.bz2

# PacBio long reads
!curl -L https://github.com/inumanag/fall25-csc-bioinf/raw/refs/heads/main/week4/data/pacbio.fq.bz2 -o pacbio.fq.bz2


  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
  0     0    0     0    0     0      0      0 --:--:-- --:--:-- --:--:--     0
100 20.7M  100 20.7M    0     0  12.7M      0  0:00:01  0:00:01 --:--:-- 55.6M
  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
  0     0    0     0    0     0      0      0 --:--:-- --:--:-- --:--:--     0
100 3253k  100 3253k    0     0  2919k      0  0:00:01  0:00:01 --:--:-- 2919k


In [9]:
!bunzip2 -k illumina.fq.bz2
!bunzip2 -k pacbio.fq.bz2


bunzip2: Output file illumina.fq already exists.
bunzip2: Output file pacbio.fq already exists.


In [10]:
!seqkit stats -a illumina.fq pacbio.fq


processed files:  1 / 2 [==================>-------------------] ETA: 0s
processed files:  2 / 2 [======================================] ETA: 0s
processed files:  2 / 2 [======================================] ETA: 0s. done
processed files:  2 / 2 [======================================] ETA: 0s. done
file         format  type  num_seqs     sum_len  min_len  avg_len  max_len     Q1     Q2       Q3  sum_gap    N50  N50_num  Q20(%)  Q30(%)  AvgQual  GC(%)  sum_n
illumina.fq  FASTQ   DNA    309,505  46,425,750      150      150      150    150    150      150        0    150        1   94.44   87.99    23.63  38.87  8,035
pacbio.fq    FASTQ   DNA      3,063  11,237,119      770  3,668.7    9,517  2,879  3,464  4,225.5        0  3,783      891    99.6   99.11    33.06  37.96      0


In [11]:
# Short-read Illumina, use sr preset
!minimap2 -ax sr chr10.fa illumina.fq | samtools sort -o illumina.bam
!samtools index illumina.bam


[M::mm_idx_gen::2.001*0.98] collected minimizers
[M::mm_idx_gen::2.314*1.25] sorted minimizers
[M::main::2.314*1.25] loaded/built the index for 1 target sequence(s)
[M::mm_mapopt_update::2.314*1.25] mid_occ = 1000
[M::mm_idx_stat] kmer size: 21; skip: 11; is_hpc: 0; #seq: 1
[M::mm_idx_stat::2.440*1.23] distinct minimizers: 19023001 (96.65% are singletons); average occurrences: 1.171; average spacing: 6.004; total length: 133797422
[M::worker_pipeline::9.180*2.46] mapped 309505 sequences
[M::main] Version: 2.30-r1287
[M::main] CMD: minimap2 -ax sr chr10.fa illumina.fq
[M::main] Real time: 9.221 sec; CPU: 22.594 sec; Peak RSS: 1.080 GB


In [12]:
# PacBio CLR – use map-pb
!minimap2 -ax map-pb chr10.fa pacbio.fq | samtools sort -o pacbio.bam
!samtools index pacbio.bam


[M::mm_idx_gen::1.922*0.99] collected minimizers
[M::mm_idx_gen::2.239*1.27] sorted minimizers
[M::main::2.239*1.27] loaded/built the index for 1 target sequence(s)
[M::mm_mapopt_update::2.347*1.26] mid_occ = 128
[M::mm_idx_stat] kmer size: 19; skip: 10; is_hpc: 1; #seq: 1
[M::mm_idx_stat::2.412*1.25] distinct minimizers: 12148713 (83.27% are singletons); average occurrences: 1.403; average spacing: 7.851; total length: 133797422
[M::worker_pipeline::3.483*1.76] mapped 3063 sequences
[M::main] Version: 2.30-r1287
[M::main] CMD: minimap2 -ax map-pb chr10.fa pacbio.fq
[M::main] Real time: 3.514 sec; CPU: 6.164 sec; Peak RSS: 0.972 GB


In [13]:
!ls -lh illumina.bam pacbio.bam illumina.bam.bai pacbio.bam.bai


-rw-r--r--  1 lev  staff    20M Nov  3 09:33 illumina.bam
-rw-r--r--  1 lev  staff    79K Nov  3 09:33 illumina.bam.bai
-rw-r--r--  1 lev  staff   2.7M Nov  3 09:33 pacbio.bam
-rw-r--r--  1 lev  staff    59K Nov  3 09:33 pacbio.bam.bai


In [14]:
!bcftools mpileup -f chr10.fa illumina.bam | bcftools call -mv -Oz -o illumina.vcf.gz
!bcftools index illumina.vcf.gz


Note: none of --samples-file, --ploidy or --ploidy-file given, assuming all sites are diploid
[mpileup] 1 samples in 1 input files
[mpileup] maximum number of reads per input file set to -d 250


In [15]:
!bcftools mpileup -f chr10.fa pacbio.bam | bcftools call -mv -Oz -o pacbio.vcf.gz
!bcftools index pacbio.vcf.gz


Note: none of --samples-file, --ploidy or --ploidy-file given, assuming all sites are diploid
[mpileup] 1 samples in 1 input files
[mpileup] maximum number of reads per input file set to -d 250


In [16]:
regions = "chr10:94759681-94853205,chr10:94938683-94993415,chr10:94842701-94924000"

!bcftools view -r {regions} -Oz -o illumina_cyp.vcf.gz illumina.vcf.gz
!bcftools index illumina_cyp.vcf.gz

!bcftools view -r {regions} -Oz -o pacbio_cyp.vcf.gz pacbio.vcf.gz
!bcftools index pacbio_cyp.vcf.gz


In [17]:
# Illumina phasing (already ran fine, but harmless if it reruns)
!whatshap phase --ignore-read-groups --reference chr10.fa \
    -o illumina_cyp_phased.vcf \
    illumina_cyp.vcf.gz illumina.bam

print("\nPhasing Illumina done.\n")

# PacBio phasing – NO --longreads flag for this WhatsHap version
!whatshap phase --ignore-read-groups --reference chr10.fa \
    -o pacbio_cyp_phased.vcf \
    pacbio_cyp.vcf.gz pacbio.bam

print("\nPhasing PacBio done.\n")

# Compress + index phased VCFs
!bgzip -f illumina_cyp_phased.vcf
!bcftools index -f illumina_cyp_phased.vcf.gz

!bgzip -f pacbio_cyp_phased.vcf
!bcftools index -f pacbio_cyp_phased.vcf.gz

!ls -lh *cyp_phased.vcf.gz

This is WhatsHap 2.8 running under Python 3.11.14
Working on 1 sample from 1 family

# Working on contig chr10 in individual illumina.bam
Found 212 usable heterozygous variants (0 skipped due to missing genotypes)
Number of supplementary alignments: 0
Number of non-singleton groups: 573
Skipped 0 groups
Found 7895 reads covering 212 variants
Kept 1175 reads that cover at least two variants each
Selected 647 most phase-informative reads covering 142 variants
Best-case phasing would result in 49 non-singleton phased blocks (0 singletons). 
Phasing 1 sample by solving the MEC problem ...
Largest block contains 9 variants (6.3% of accessible variants) between position 94896031 and 94898738

# Resource usage
Time spent reading BAM/CRAM:                    1.7 s
Time spent parsing VCF:                         0.0 s
Time spent selecting reads:                     0.0 s
Time spent phasing:                             0.2 s
Time spent writing VCF:                         0.0 s
Time spent findin

In [18]:
!mkdir -p isec_output
!bcftools isec -p isec_output illumina_cyp_phased.vcf.gz pacbio_cyp_phased.vcf.gz

print("Illumina-only variants:")
!wc -l isec_output/0000.vcf

print("\nPacBio-only variants:")
!wc -l isec_output/0001.vcf

print("\nShared variants:")
!wc -l isec_output/0002.vcf


Illumina-only variants:
     162 isec_output/0000.vcf

PacBio-only variants:
      78 isec_output/0001.vcf

Shared variants:
     220 isec_output/0002.vcf


In [19]:
print("CYP2C19 variant count (Illumina, phased):")
!bcftools view -H -r chr10:94759681-94853205 illumina_cyp_phased.vcf.gz | wc -l

print("\nCYP2C9 variant count (Illumina, phased):")
!bcftools view -H -r chr10:94938683-94993415 illumina_cyp_phased.vcf.gz | wc -l

print("\nCYP2C8 variant count (Illumina, phased):")
!bcftools view -H -r chr10:94842701-94924000 illumina_cyp_phased.vcf.gz | wc -l


CYP2C19 variant count (Illumina, phased):
     125

CYP2C9 variant count (Illumina, phased):
      71

CYP2C8 variant count (Illumina, phased):
     129


In [20]:
print("=== CYP2C19 variants (Illumina, phased) ===")
!bcftools query -f '%POS\t%REF\t%ALT\t[%GT]\n' -r chr10:94759681-94853205 illumina_cyp_phased.vcf.gz | head -n 50


=== CYP2C19 variants (Illumina, phased) ===
94760573	C	A	0/1
94761900	C	T	0/1
94762804	C	T	1/1
94763232	A	T	0/1
94765362	G	A	0/1
94765779	G	A	1/1
94766304	ttg	tTGTAtg	1/1
94766352	G	A	0/1
94767381	A	C	1/1
94767577	C	A	0/1
94767965	C	G	1/1
94768394	G	A	0/1
94770155	A	G	1/1
94772788	G	T	1/1
94772850	T	C	1|0
94772907	G	A	1|0
94772931	G	A	0|1
94773293	A	T	1/1
94774827	G	A	1/1
94775961	G	A	0/1
94776698	T	A	0|1
94776951	C	T	0|1
94777632	A	G	0/1
94778957	T	A	1|0
94778981	G	A	0|1
94779551	tttttcttttcttttctttt	tTTTTCTTTTCttttcttttcttttctttt	0|1
94782837	C	T	1|0
94783106	A	G	1|0
94784235	cttttt	cTttttt	1/1
94784708	G	A	1|0
94784970	C	G	1|0
94785981	C	T	0/1
94787917	G	A	0/1
94788106	gttccattcca	gttcca	0|1
94788559	T	C	0|1
94789209	A	G	0|1
94791495	G	T	0/1
94792552	G	T	0/1
94793025	A	C	1/1
94794531	A	T	1/1
94794659	A	G	1/1
94795304	T	G	0|1
94795486	C	A	0|1
94795832	C	G	0|1
94797351	T	C	0/1
94798522	C	G	1|0
94798814	atttttttttttttttttttttttt	atttttttttttttttttttttt	0|1
94801232	C	T	0|1
94801768	A	G

In [21]:
print("\n=== CYP2C9 variants (Illumina, phased) ===")
!bcftools query -f '%POS\t%REF\t%ALT\t[%GT]\n' -r chr10:94938683-94993415 illumina_cyp_phased.vcf.gz | head -n 50



=== CYP2C9 variants (Illumina, phased) ===
94938933	T	C	0/1
94939926	G	A	0/1
94940780	C	T	1/1
94940873	A	T	1/1
94941267	gtttttttttttttttttttttttttt	gTTtttttttttttttttttttttttttt	1/1
94941320	G	T	0/1
94942093	T	C	0/1
94943734	C	A	0/1
94944819	A	G	1/1
94945466	A	T	0/1
94946130	T	C	1|0
94946131	G	T	1|0
94946177	A	G	1/1
94946652	C	G	0/1
94950236	T	C	1/1
94950968	C	T	1/1
94952127	G	A	0|1
94952386	T	C	1|0
94952643	A	G	1/1
94952785	tatagata	tATAGatagata	0|1
94953443	T	C	0/1
94954003	G	A	0/1
94955914	T	C	1|0
94956377	C	T	1|0
94956617	A	C	1|0
94957439	T	G	1/1
94958456	T	C	1|0
94959037	C	T	1|0
94959052	G	T	0|1
94959431	A	G	1/1
94959719	A	G	1/1
94960070	A	G	1/1
94960275	T	C	1/1
94960761	A	G	0/1
94961350	G	C	1/1
94962302	C	A	1/1
94963667	A	G	0/1
94964413	T	C	1/1
94966538	tcccc	tccc	0/1
94967800	G	A	1|0
94968273	C	T	1|0
94968463	A	C	1|0
94968568	T	G	1|0
94968623	gttttttttttt	gtttttttttt	0|1
94969284	A	C	0/1
94970667	G	A	1|0
94971060	T	A	0|1
94971535	C	T	1|0
94972340	A	G	0/1
94972974	T	G	1/1


In [22]:
print("\n=== CYP2C8 variants (Illumina, phased) ===")
!bcftools query -f '%POS\t%REF\t%ALT\t[%GT]\n' -r chr10:94842701-94924000 illumina_cyp_phased.vcf.gz | head -n 50



=== CYP2C8 variants (Illumina, phased) ===
94842866	A	G	1/1
94843657	A	G	0|1
94844516	C	G	0|1
94845570	G	C	1/1
94848458	C	T	0/1
94849307	G	A	1/1
94850760	T	C	1/1
94851443	Caaataaataaataaataaataaataaataaataaataaataaataaataa	Caaataaataaataaataaataaataaataaataaataaataaataa	0|1
94851871	T	C	0|1
94852077	G	A	1/1
94852283	G	A	0|1
94852614	C	T	0|1
94852716	A	G	1|0
94853547	Ctttttttttttttttt	CTtttttttttttttttt	1/1
94854102	tccacc	tcc	0/1
94854968	G	T	0/1
94856167	C	T	0/1
94856660	A	T	1/1
94857274	T	G	0/1
94858114	A	G	0/1
94859474	C	T	0/1
94859761	GCCCCCCCC	GCCCCCCCCCC,GCCCCCCCCC	1/2
94859849	G	A	1/1
94860797	A	G	1/1
94860863	caaaaaaaaa	cAaaaaaaaaa	0/1
94861449	G	C	1/1
94861942	T	C	0/1
94864151	gtttttttttttt	gTtttttttttttt	0/1
94866231	C	T	0|1
94866445	T	C	1|0
94866498	G	A	1|0
94868517	T	C	0|1
94868852	C	A	0|1
94869480	A	C	1|0
94870493	caaaaaaaaaaaaaa	caaaaaaaaaaaaa	0|1
94870503	A	T	1|0
94872496	T	C	1/1
94873074	A	G	0/1
94874084	C	T	0/1
94876348	T	C	0/1
94877910	A	G	1/1
94878357	C	T	0/1
948796

In [23]:
print("Checking key PharmVar-style markers in Illumina CYP2C VCF\n")

print("CYP2C19*2 (rs4244285, ~chr10:94781859):")
!bcftools view -r chr10:94781859-94781859 illumina_cyp_phased.vcf.gz | \
  bcftools query -f '%CHROM\t%POS\t%REF\t%ALT\t[%GT]\n' || echo "not found"

print("\nCYP2C19*17 (rs12248560, ~chr10:94761900):")
!bcftools view -r chr10:94761900-94761900 illumina_cyp_phased.vcf.gz | \
  bcftools query -f '%CHROM\t%POS\t%REF\t%ALT\t[%GT]\n' || echo "not found"

print("\nCYP2C9*2 (rs1799853, ~chr10:94942290):")
!bcftools view -r chr10:94942290-94942290 illumina_cyp_phased.vcf.gz | \
  bcftools query -f '%CHROM\t%POS\t%REF\t%ALT\t[%GT]\n' || echo "not found"

print("\nCYP2C9*3 (rs1057910, ~chr10:94981296):")
!bcftools view -r chr10:94981296-94981296 illumina_cyp_phased.vcf.gz | \
  bcftools query -f '%CHROM\t%POS\t%REF\t%ALT\t[%GT]\n' || echo "not found"

print("\nCYP2C8*3 markers (rs11572080 ~95038992, rs10509681 ~95067273):")
!bcftools view -r chr10:95038992-95038992 illumina_cyp_phased.vcf.gz | \
  bcftools query -f 'rs11572080?\t%CHROM\t%POS\t%REF\t%ALT\t[%GT]\n' || echo "rs11572080 not found"
!bcftools view -r chr10:95067273-95067273 illumina_cyp_phased.vcf.gz | \
  bcftools query -f 'rs10509681?\t%CHROM\t%POS\t%REF\t%ALT\t[%GT]\n' || echo "rs10509681 not found"


Checking key PharmVar-style markers in Illumina CYP2C VCF

CYP2C19*2 (rs4244285, ~chr10:94781859):

CYP2C19*17 (rs12248560, ~chr10:94761900):
chr10	94761900	C	T	0/1

CYP2C9*2 (rs1799853, ~chr10:94942290):

CYP2C9*3 (rs1057910, ~chr10:94981296):

CYP2C8*3 markers (rs11572080 ~95038992, rs10509681 ~95067273):
